# **Orange Vision: Detección de Naranjas con YOLOv11**
### Flujo integral: Inferencia Pura, Transfer Learning, Fine-Tuning y Detección en Tiempo Real (Webcam)
Este cuaderno implementa el ciclo completo de visión artificial para la detección de naranjas:
1. **Inferencia Pura**: Utilización del modelo base YOLOv11 preentrenado en el dataset COCO (filtrando la clase `orange`).
2. **Dataset "Oranges in the field" (Mendeley Data)**: Descarga y estructuración automática de imágenes de campo con etiquetas YOLO (https://data.mendeley.com/datasets/93f32zgkxz/1).
3. **Transfer Learning**: Congelamiento de las primeras 10 capas (*backbone*) y reentrenamiento de la cabeza detectora con el dataset de naranjas.
4. **Fine-Tuning**: Descongelamiento completo de la red y entrenamiento fino con baja tasa de aprendizaje (*Learning Rate* adaptativo con AdamW).
5. **Comparación y Métricas**: Evaluación sobre el conjunto de test con mAP50, mAP50-95 y Recall comparando el modelo base vs transfer learning vs fine-tuning.
6. **Detección con Foto**: Inferencia sobre imágenes estáticas comparando modelos.
7. **Detección en Tiempo Real con Webcam**: Procesamiento continuo de video con OpenCV, conteo de naranjas y medición de FPS.

In [16]:
# 1. Librerías Básicas
%pip install -q ultralytics torchmetrics pycocotools opencv-python

import os
import shutil
import time
import json
import random
import urllib.request
import joblib
import numpy as np
import pandas as pd
import torch
import cv2
from pathlib import Path

from ultralytics import YOLO, settings
from ultralytics.data.utils import check_det_dataset

# Métricas de detección
from torchmetrics.detection import MeanAveragePrecision

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\sanie\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [17]:
# 2. Configurar clases de interés para Inferencia Pura
# El modelo YOLO preentrenado en COCO incluye la clase 'orange' (ID 49 en COCO).
class_names = ['orange']
print('Clases objetivo iniciales para detección:', class_names)

Clases objetivo iniciales para detección: ['orange']


In [18]:
# 3. Inferencia pura: YOLO preentrenado en COCO, sin entrenar con nuestros datos
modelo_base = YOLO('yolo11n.pt')

print('Total de clases que conoce YOLO COCO:', len(modelo_base.names))
for nombre in class_names:
    print(f'  {nombre}:', 'sí' if nombre in modelo_base.names.values() else 'NO')

Total de clases que conoce YOLO COCO: 80
  orange: sí


In [19]:
# 4. Función para realizar predicciones filtrando por 'orange' (naranja)
def detectar_naranjas(modelo, ruta_imagen, conf_min=0.25):
    """
    Realiza inferencia sobre una imagen y muestra las detecciones de naranjas.
    """
    ruta_p = Path(ruta_imagen)
    if not ruta_p.exists():
        print(f"Error: La imagen '{ruta_imagen}' no existe.")
        return None

    # Inferencia con YOLO
    resultados = modelo.predict(ruta_imagen, conf=conf_min, verbose=False)[0]

    nombre_modelo = getattr(modelo, 'model_name', 'YOLO')
    print(f"\nResultados para {ruta_p.name} ({nombre_modelo}):")
    encontrado = False
    conteo = 0

    for caja in resultados.boxes:
        nombre_clase = resultados.names[int(caja.cls)]
        # Si el modelo tiene la clase 'orange', o si tiene solo 1 clase entrenada (dataset propio)
        if nombre_clase == 'orange' or len(resultados.names) == 1:
            encontrado = True
            conteo += 1
            conf = round(float(caja.conf), 3)
            coords = [round(x, 1) for x in caja.xyxy[0].tolist()]
            print(f"  [Naranja #{conteo}] Confianza: {conf} | Coordenadas: {coords}")

    if not encontrado:
        print("  No se detectaron naranjas en la imagen con la confianza mínima requerida.")
    else:
        print(f"  Total de naranjas detectadas: {conteo}")

    resultados.show()
    return resultados

In [20]:
# 5. Detección con una foto usando el modelo base
ruta_prueba = 'orange_sample.jpg'

# Si no existe una foto local, generamos una imagen sintética de prueba
if not Path(ruta_prueba).exists():
    img_temp = np.zeros((480, 640, 3), dtype=np.uint8)
    img_temp[:] = (34, 110, 34)  # Fondo verdoso simulando follaje
    cv2.circle(img_temp, (240, 240), 75, (0, 140, 255), -1)  # Naranja 1
    cv2.circle(img_temp, (420, 210), 65, (0, 165, 255), -1)  # Naranja 2
    cv2.circle(img_temp, (330, 330), 80, (0, 150, 255), -1)  # Naranja 3
    cv2.imwrite(ruta_prueba, img_temp)
    print(f"Se ha creado una imagen de prueba temporal en: {ruta_prueba}")

detectar_naranjas(modelo_base, ruta_prueba, conf_min=0.25)


Resultados para orange_sample.jpg (yolo11n.pt):
  [Naranja #1] Confianza: 0.908 | Coordenadas: [148.5, 37.2, 240.9, 135.5]
  [Naranja #2] Confianza: 0.83 | Coordenadas: [98.3, 104.9, 181.1, 187.4]
  [Naranja #3] Confianza: 0.373 | Coordenadas: [217.9, 93.8, 249.9, 139.4]
  Total de naranjas detectadas: 3


ultralytics.engine.results.Results object with attributes:

boxes: ultralytics.engine.results.Boxes object
depth: None
keypoints: None
masks: None
names: {0: 'person', 1: 'bicycle', 2: 'car', 3: 'motorcycle', 4: 'airplane', 5: 'bus', 6: 'train', 7: 'truck', 8: 'boat', 9: 'traffic light', 10: 'fire hydrant', 11: 'stop sign', 12: 'parking meter', 13: 'bench', 14: 'bird', 15: 'cat', 16: 'dog', 17: 'horse', 18: 'sheep', 19: 'cow', 20: 'elephant', 21: 'bear', 22: 'zebra', 23: 'giraffe', 24: 'backpack', 25: 'umbrella', 26: 'handbag', 27: 'tie', 28: 'suitcase', 29: 'frisbee', 30: 'skis', 31: 'snowboard', 32: 'sports ball', 33: 'kite', 34: 'baseball bat', 35: 'baseball glove', 36: 'skateboard', 37: 'surfboard', 38: 'tennis racket', 39: 'bottle', 40: 'wine glass', 41: 'cup', 42: 'fork', 43: 'knife', 44: 'spoon', 45: 'bowl', 46: 'banana', 47: 'apple', 48: 'sandwich', 49: 'orange', 50: 'broccoli', 51: 'carrot', 52: 'hot dog', 53: 'pizza', 54: 'donut', 55: 'cake', 56: 'chair', 57: 'couch', 58: 'po

In [21]:
# 6. Cargar datos: Dataset "Oranges in the field" (Mendeley Data)
# Referencia: https://data.mendeley.com/datasets/93f32zgkxz/1
# 5025 imágenes de 640x640 tomadas en campos de Sicilia con etiquetas YOLO (clase xc yc w h).

API_NARANJAS = 'https://data.mendeley.com/public-api/datasets/93f32zgkxz'
CABECERA = {'User-Agent': 'Mozilla/5.0'}
CARPETA_DATASET = Path('dataset_naranjas')
USAR_CARPETA_LOCAL = False  # Cambiar a True si ya tienes el dataset descargado en local

class_names = ['orange']

def descargar_dataset_mendeley(n_imagenes=120, carpeta_destino='dataset_naranjas', 
                               split_train=0.7, split_val=0.2, semilla=42, prefijo=''):
    """
    Descarga imágenes y etiquetas YOLO desde Mendeley Data y organiza las carpetas en
    formato estándar de YOLO (images/train, images/val, images/test, labels/...).
    """
    destino = Path(carpeta_destino)
    yaml_path = destino / 'data.yaml'

    # Verificar si ya fue descargado previamente
    imgs_descargadas = list(destino.glob('images/**/*.jpg'))
    if len(imgs_descargadas) >= int(n_imagenes * 0.8) and yaml_path.exists():
        print(f"Dataset ya encontrado en '{destino}' con {len(imgs_descargadas)} imágenes.")
        return yaml_path

    print(f"Consultando API de Mendeley Data ({API_NARANJAS})...")
    req = urllib.request.Request(API_NARANJAS, headers=CABECERA)
    with urllib.request.urlopen(req, timeout=45) as r:
        datos = json.loads(r.read().decode('utf-8'))

    archivos = datos['files']
    imagenes = sorted([a for a in archivos if a['filename'].endswith('.jpg') and a['filename'].startswith(prefijo)],
                      key=lambda x: x['filename'])
    etiquetas = {a['filename'][:-4]: a for a in archivos if a['filename'].endswith('.txt')}

    random.Random(semilla).shuffle(imagenes)
    seleccion = imagenes[:n_imagenes]

    n_train = int(len(seleccion) * split_train)
    n_val = int(len(seleccion) * split_val)

    splits = {
        'train': seleccion[:n_train],
        'val': seleccion[n_train:n_train + n_val],
        'test': seleccion[n_train + n_val:]
    }

    print(f"Descargando {len(seleccion)} imágenes de naranjas (Train: {len(splits['train'])}, Val: {len(splits['val'])}, Test: {len(splits['test'])})...")

    for split_name, lista_imgs in splits.items():
        dir_img = destino / 'images' / split_name
        dir_lbl = destino / 'labels' / split_name
        dir_img.mkdir(parents=True, exist_ok=True)
        dir_lbl.mkdir(parents=True, exist_ok=True)

        for item in lista_imgs:
            base = item['filename'][:-4]
            ruta_img = dir_img / item['filename']
            ruta_lbl = dir_lbl / f"{base}.txt"

            # Descargar imagen JPG
            if not ruta_img.exists() or ruta_img.stat().st_size == 0:
                try:
                    req_img = urllib.request.Request(item['content_details']['download_url'], headers=CABECERA)
                    with urllib.request.urlopen(req_img, timeout=45) as r_img:
                        with open(ruta_img, 'wb') as f_img:
                            f_img.write(r_img.read())
                except Exception as e:
                    print(f"Error descargando {item['filename']}: {e}")

            # Descargar etiqueta TXT (formato YOLO)
            if base in etiquetas and (not ruta_lbl.exists() or ruta_lbl.stat().st_size == 0):
                try:
                    req_lbl = urllib.request.Request(etiquetas[base]['content_details']['download_url'], headers=CABECERA)
                    with urllib.request.urlopen(req_lbl, timeout=45) as r_lbl:
                        with open(ruta_lbl, 'wb') as f_lbl:
                            f_lbl.write(r_lbl.read())
                except Exception as e:
                    print(f"Error descargando etiqueta {base}.txt: {e}")

    # Generar data.yaml para Ultralytics YOLO
    with open(yaml_path, 'w', encoding='utf-8') as f:
        f.write(f"path: {destino.resolve().as_posix()}\n")
        f.write("train: images/train\n")
        f.write("val: images/val\n")
        f.write("test: images/test\n")
        f.write("names:\n")
        f.write("  0: orange\n")

    print(f"data.yaml generado en: {yaml_path}")
    return yaml_path

# Preparar y cargar el dataset
if USAR_CARPETA_LOCAL and CARPETA_DATASET.exists():
    yaml_path = CARPETA_DATASET / 'data.yaml'
    DATA = str(yaml_path)
else:
    # Puedes ajustar n_imagenes (ej. 100 para prueba rápida o más para entrenamiento intensivo)
    yaml_path = descargar_dataset_mendeley(n_imagenes=120, carpeta_destino='dataset_naranjas', semilla=42)
    DATA = str(yaml_path)

info = check_det_dataset(DATA)
TEST = info.get('test', info.get('val'))
print('Dataset listo:', DATA)
print('Clases:', class_names)
print('Imágenes de test:', len(list(Path(TEST).glob('*.jpg'))) if Path(TEST).exists() else 'N/A')

Dataset ya encontrado en 'dataset_naranjas' con 119 imágenes.
Dataset listo: dataset_naranjas\data.yaml
Clases: ['orange']
Imágenes de test: 12


In [22]:
# 7. Configurar TensorBoard para monitoreo del entrenamiento
%load_ext tensorboard

settings.update({'tensorboard': True})

# Limpiar ejecuciones previas de transfer y fine-tuning si se desea reiniciar
shutil.rmtree('runs/detect/transfer_orange', ignore_errors=True)
shutil.rmtree('runs/detect/finetune_orange', ignore_errors=True)

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [23]:
# 8. Transfer Learning: Modelo preentrenado con las primeras 10 capas congeladas
# Solo se entrenan las capas finales (head), adaptándolas al dataset de naranjas de Mendeley.
modelo_transfer = YOLO('yolo11n.pt')

modelo_transfer.train(
    data=DATA,
    epochs=10,
    freeze=10,
    patience=5,
    imgsz=640,
    name='transfer_orange'
)

New https://pypi.org/project/ultralytics/8.4.177 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.175  Python-3.13.14 torch-2.14.1+cpu CPU (12th Gen Intel Core i5-12450H)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset_naranjas\data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=

C:\Users\sanie\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\torch\jit\_trace.py:1006: FutureWarning: `torch.jit.trace` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
C:\Users\sanie\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\torch\jit\_trace.py:1145: FutureWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


TensorBoard: model graph visualization added 
Using 84 train, 23 val images for fraction=1.0 at imgsz=640
Using 0 dataloader workers
Logging results to C:\Users\sanie\Documents\GitHub\protectoIA2026_orange_vision\entrenamiento\runs\detect\transfer_orange
Starting training for 10 epochs...
Closing dataloader mosaic


C:\Users\sanie\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\torch\jit\_trace.py:1022: UserWarning: The input to trace is already a ScriptModule, tracing it is a no-op. Returning the object as is.
  traced_func = _trace_impl(



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       1/10         0G      1.832      2.685      1.366         48        640: 100% ━━━━━━━━━━━━ 6/6 3.2s/it 19.0s2.3ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 2.8s/it 2.8s
                   all         23        280      0.703      0.486      0.543      0.256

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/10         0G      1.988      1.602       1.49         10        640: 100% ━━━━━━━━━━━━ 6/6 2.9s/it 17.3s1.9ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 3.4s/it 3.4s
                   all         23        280      0.229      0.532      0.328      0.121

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/10         0G      2.172       1.66      1.541         37        640: 100% ━━━━━━━━━━━

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000001D92970B0E0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0480

In [24]:
# 9. Fine-Tuning: Partir del modelo de transfer, descongelar todas las capas y reentrenar con LR bajo
# Se reentrena la red completa con un optimizador adaptativo y learning rate pequeño (lr0=1e-4)
# para refinar las características de bajo y alto nivel sin olvidar lo aprendido.
modelo_ft = YOLO(modelo_transfer.trainer.best)

modelo_ft.train(
    data=DATA,
    epochs=10,
    optimizer='AdamW',
    lr0=1e-4,
    warmup_epochs=0,
    freeze=0,
    patience=5,
    imgsz=640,
    name='finetune_orange'
)

New https://pypi.org/project/ultralytics/8.4.177 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.175  Python-3.13.14 torch-2.14.1+cpu CPU (12th Gen Intel Core i5-12450H)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset_naranjas\data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=0, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=C:\Users\sanie\Docum

C:\Users\sanie\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\torch\jit\_trace.py:1006: FutureWarning: `torch.jit.trace` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
C:\Users\sanie\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\torch\jit\_trace.py:1145: FutureWarning: `torch.jit.trace_method` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


TensorBoard: model graph visualization added 
Using 84 train, 23 val images for fraction=1.0 at imgsz=640
Using 0 dataloader workers
Logging results to C:\Users\sanie\Documents\GitHub\protectoIA2026_orange_vision\entrenamiento\runs\detect\finetune_orange
Starting training for 10 epochs...
Closing dataloader mosaic


C:\Users\sanie\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\torch\jit\_trace.py:1022: UserWarning: The input to trace is already a ScriptModule, tracing it is a no-op. Returning the object as is.
  traced_func = _trace_impl(



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       1/10         0G      1.721      1.724      1.288         48        640: 100% ━━━━━━━━━━━━ 6/6 4.9s/it 29.3s3.3sss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 2.8s/it 2.8s
                   all         23        280        0.7      0.539      0.571      0.269

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/10         0G      1.838       1.52      1.403         10        640: 100% ━━━━━━━━━━━━ 6/6 4.3s/it 25.6s3.0ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 2.8s/it 2.8s
                   all         23        280      0.683      0.523      0.563      0.272

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/10         0G      1.686       1.25      1.282         37        640: 100% ━━━━━━━━━━

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x000001D9294BB700>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0480

In [25]:
# 10. Visualizar y contrastar las curvas de pérdida y métricas en TensorBoard
%tensorboard --logdir runs

Reusing TensorBoard on port 6006 (pid 24632), started 0:13:29 ago. (Use '!kill 24632' to kill it.)

In [27]:
# 11. Evaluar y comparar los tres modelos sobre el set de test de Mendeley
# Comparamos Inferencia pura (COCO), Transfer Learning y Fine-Tuning sobre las mismas fotos de campo.

def evaluar(modelo):
    mapa = {i: class_names.index(n) for i, n in modelo.names.items() if n in class_names}
    metrica = MeanAveragePrecision(class_metrics=True)

    if not TEST or not Path(TEST).exists():
        return {'mAP50': 0.0, 'mAP50-95': 0.0, 'Recall': 0.0}

    for r in modelo.predict(TEST, conf=0.001, max_det=100, stream=True, verbose=False):
        cls = r.boxes.cls.int().tolist()
        ok = [k for k, c in enumerate(cls) if c in mapa]
        pred = {
            'boxes': r.boxes.xyxy[ok].cpu(),
            'scores': r.boxes.conf[ok].cpu(),
            'labels': torch.tensor([mapa[cls[k]] for k in ok], dtype=torch.long)
        }

        h, w = r.orig_shape
        # Reemplazar path a formato label
        txt_path = Path(r.path.replace('/images/', '/labels/').replace('\\images\\', '\\labels\\')).with_suffix('.txt')
        e = np.loadtxt(txt_path, ndmin=2) if txt_path.exists() and txt_path.read_text().strip() else np.zeros((0, 5))
        xyxy = np.c_[e[:, 1] - e[:, 3] / 2, e[:, 2] - e[:, 4] / 2,
                     e[:, 1] + e[:, 3] / 2, e[:, 2] + e[:, 4] / 2] * [w, h, w, h]
        real = {
            'boxes': torch.tensor(xyxy, dtype=torch.float32),
            'labels': torch.tensor(e[:, 0], dtype=torch.long)
        }

        metrica.update([pred], [real])

    m = metrica.compute()
    fila = {
        'mAP50': float(m['map_50']),
        'mAP50-95': float(m['map']),
        'Recall': float(m['mar_100'])
    }
    
    classes_t = m['classes']
    map_pc_t  = m['map_per_class']
    if getattr(classes_t, 'ndim', 0) > 0:
        for c, v in zip(classes_t.tolist(), map_pc_t.tolist()):
            if c < len(class_names):
                fila[f'mAP {class_names[c]}'] = max(float(v), 0.0)
    return fila


# Seleccionar los mejores checkpoints obtenidos
try:
    mejor_transfer = YOLO(modelo_transfer.trainer.best)
except Exception:
    mejor_transfer = modelo_base

try:
    modelo_final = YOLO(modelo_ft.trainer.best)
except Exception:
    modelo_final = mejor_transfer

tabla_comparacion = pd.DataFrame({
    'Inferencia pura (COCO)': evaluar(modelo_base),
    'Transfer Learning (Mendeley)': evaluar(mejor_transfer),
    'Fine-Tuning (Mendeley)': evaluar(modelo_final),
}).T.round(3)

print("=== TABLA DE COMPARACIÓN DE DESEMPEÑO ===")
tabla_comparacion

=== TABLA DE COMPARACIÓN DE DESEMPEÑO ===


,mAP50,mAP50-95,Recall
Inferencia pura (COCO),0.526,0.279,0.369
Transfer Learning (Mendeley),0.628,0.383,0.502
Fine-Tuning (Mendeley),0.714,0.421,0.556


In [28]:
# 12. Comparar visualmente la detección en una foto de prueba
# Comparación directa entre Inferencia Pura y el Modelo Final ajustado
foto_evaluacion = ruta_prueba

modelos_comparar = [
    ('Modelo Base (Inferencia Pura COCO)', modelo_base),
    ('Modelo Final (Fine-Tuning Mendeley)', modelo_final)
]

for nombre, mod in modelos_comparar:
    print(f"\n==================== {nombre} ====================")
    detectar_naranjas(mod, foto_evaluacion, conf_min=0.25)


==================== Modelo Base (Inferencia Pura COCO) ====================

Resultados para orange_sample.jpg (yolo11n.pt):
  [Naranja #1] Confianza: 0.908 | Coordenadas: [148.5, 37.2, 240.9, 135.5]
  [Naranja #2] Confianza: 0.83 | Coordenadas: [98.3, 104.9, 181.1, 187.4]
  [Naranja #3] Confianza: 0.373 | Coordenadas: [217.9, 93.8, 249.9, 139.4]
  Total de naranjas detectadas: 3

==================== Modelo Final (Fine-Tuning Mendeley) ====================

Resultados para orange_sample.jpg (C:\Users\sanie\Documents\GitHub\protectoIA2026_orange_vision\entrenamiento\runs\detect\finetune_orange\weights\best.pt):
  [Naranja #1] Confianza: 0.963 | Coordenadas: [146.4, 34.4, 242.7, 140.6]
  [Naranja #2] Confianza: 0.799 | Coordenadas: [99.7, 103.3, 183.8, 189.5]
  [Naranja #3] Confianza: 0.521 | Coordenadas: [215.4, 91.1, 250.0, 140.7]
  [Naranja #4] Confianza: 0.374 | Coordenadas: [0.0, 175.9, 20.6, 210.6]
  Total de naranjas detectadas: 4


In [29]:
# 13. Guardar el modelo final y sus metadatos
archivo_modelo = 'orange_detector_best.pt'
modelo_final.save(archivo_modelo)

# Guardar clases y configuraciones adicionales
joblib.dump({
    'class_names': class_names,
    'source_base': 'yolo11n.pt',
    'dataset': 'Mendeley Data - Oranges in the field'
}, 'orange_detector_extras.joblib')

print(f"Modelo final guardado en: {archivo_modelo}")
print("Metadatos guardados en: orange_detector_extras.joblib")

Modelo final guardado en: orange_detector_best.pt
Metadatos guardados en: orange_detector_extras.joblib


In [33]:
# 14. Detección en tiempo real usando Webcam (OpenCV)
# Permite detectar naranjas en vivo mediante la cámara conectada a tu computadora.
# Presiona la tecla 'q' o 'ESC' en la ventana de video para salir.

def detectar_en_webcam(modelo, cam_id=0, conf_min=0.3):
    """
    Inicia la captura de video en vivo desde la webcam, detecta naranjas en tiempo real,
    dibuja las cajas delimitadoras, el conteo actual y los FPS.
    """
    cap = cv2.VideoCapture(cam_id)
    if not cap.isOpened():
        print(f"Error: No se pudo acceder a la webcam con ID {cam_id}.")
        print("Si tienes varias cámaras, intenta cambiando cam_id (ej. 1 o 2).")
        return

    print(f"Cámara #{cam_id} iniciada con éxito.")
    print("Controles: Presiona 'q' o 'ESC' en la ventana de video para salir.")

    prev_time = time.time()

    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                print("No se pudo leer el fotograma de la cámara.")
                break

            # Inferencia con YOLO sobre el fotograma actual
            resultados = modelo.predict(frame, conf=conf_min, verbose=False)[0]

            conteo_naranjas = 0

            # Procesar las cajas detectadas
            for caja in resultados.boxes:
                cls_id = int(caja.cls[0])
                nombre_clase = resultados.names[cls_id]

                # Filtrar naranjas (o clase única del modelo entrenado)
                if nombre_clase == 'orange' or len(resultados.names) == 1:
                    conteo_naranjas += 1
                    confianza = float(caja.conf[0])
                    x1, y1, x2, y2 = map(int, caja.xyxy[0])

                    # Dibujar caja delimitadora naranja
                    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 165, 255), 2)
                    
                    # Etiqueta con porcentaje de confianza
                    texto = f"Naranja: {confianza * 100:.1f}%"
                    (tw, th), _ = cv2.getTextSize(texto, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
                    cv2.rectangle(frame, (x1, max(0, y1 - 25)), (x1 + tw, max(0, y1)), (0, 165, 255), -1)
                    cv2.putText(frame, texto, (x1, max(18, y1 - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

            # Cálculo de fotogramas por segundo (FPS)
            curr_time = time.time()
            fps = 1.0 / (curr_time - prev_time) if (curr_time - prev_time) > 0 else 0.0
            prev_time = curr_time

            # Barra informativa superior
            cv2.rectangle(frame, (0, 0), (320, 75), (0, 0, 0), -1)
            cv2.putText(frame, f"FPS: {fps:.1f}", (15, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 255, 0), 2)
            cv2.putText(frame, f"Naranjas: {conteo_naranjas}", (15, 55), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 165, 255), 2)

            # Mostrar ventana con el fotograma anotado
            cv2.imshow("Orange Vision - Deteccion en Tiempo Real", frame)

            # Salir con la tecla 'q' o 'ESC' (código 27)
            tecla = cv2.waitKey(1) & 0xFF
            if tecla == ord('q') or tecla == 27:
                print("Detección detenida por el usuario.")
                break

    finally:
        cap.release()
        cv2.destroyAllWindows()
        print("Recursos de la cámara liberados correctamente.")

# Ejecutar detección con webcam:
# Puedes usar 'modelo_final' o 'modelo_base'
modelo_a_usar = modelo_final if 'modelo_final' in locals() else modelo_base
# Para iniciar la cámara en tu máquina local, descomenta y ejecuta la siguiente línea:
detectar_en_webcam(modelo_a_usar, cam_id=0, conf_min=0.25)
print("Función detectar_en_webcam cargada. Descomenta la línea anterior para iniciar tu cámara.")

Cámara #0 iniciada con éxito.
Controles: Presiona 'q' o 'ESC' en la ventana de video para salir.
Detección detenida por el usuario.
Recursos de la cámara liberados correctamente.
Función detectar_en_webcam cargada. Descomenta la línea anterior para iniciar tu cámara.


In [32]:
# 15. [Opcional] Captura de foto con Webcam en Google Colab
# Google Colab se ejecuta en la nube y no tiene acceso directo a ventanas de cv2.imshow().
# Si ejecutas este notebook en Google Colab, esta celda captura una foto usando JavaScript
# directamente desde tu navegador web y realiza la detección de naranjas sobre ella.

from IPython.display import display, Javascript
try:
    from google.colab.output import eval_js
    import base64

    def capturar_foto_colab(filename='colab_webcam_photo.jpg', quality=0.8):
        js = Javascript('''
            async function takePhoto(quality) {
                const div = document.createElement('div');
                const capture = document.createElement('button');
                capture.textContent = 'Tomar Foto para Detección';
                capture.className = 'btn';
                div.appendChild(capture);

                const video = document.createElement('video');
                video.style.display = 'block';
                const stream = await navigator.mediaDevices.getUserMedia({video: true});

                document.body.appendChild(div);
                div.appendChild(video);
                video.srcObject = stream;
                await video.play();

                // Esperar clic en el botón
                await new Promise((resolve) => capture.onclick = resolve);

                const canvas = document.createElement('canvas');
                canvas.width = video.videoWidth;
                canvas.height = video.videoHeight;
                canvas.getContext('2d').drawImage(video, 0, 0);
                stream.getVideoTracks()[0].stop();
                div.remove();
                return canvas.toDataURL('image/jpeg', quality);
            }
        ''')
        display(js)
        data = eval_js('takePhoto({})'.format(quality))
        binary = base64.b64decode(data.split(',')[1])
        with open(filename, 'wb') as f:
            f.write(binary)
        print(f"Foto capturada con éxito en: {filename}")
        return filename

    # Ejemplo de uso en Colab:
    # foto_colab = capturar_foto_colab()
    # if foto_colab:
    #     detectar_naranjas(modelo_a_usar, foto_colab, conf_min=0.25)
except ImportError:
    print("Entorno local detectado. Para webcam en local, usa la celda anterior 'detectar_en_webcam'.")

Entorno local detectado. Para webcam en local, usa la celda anterior 'detectar_en_webcam'.
